In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 5GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/train.csv
/kaggle/input/test.csv
/kaggle/input/description.md
/kaggle/input/sample_submission.csv
/kaggle/input/train.csv.zip
/kaggle/input/test.csv.zip
/kaggle/input/sample_submission.csv.zip
/kaggle/input/tweet-sentiment-extraction/train.csv
/kaggle/input/tweet-sentiment-extraction/test.csv
/kaggle/input/tweet-sentiment-extraction/description.md
/kaggle/input/tweet-sentiment-extraction/sample_submission.csv
/kaggle/input/tweet-sentiment-extraction/train.csv.zip
/kaggle/input/tweet-sentiment-extraction/test.csv.zip
/kaggle/input/tweet-sentiment-extraction/sample_submission.csv.zip


In [2]:
import numpy as np
import pandas as pd
import tokenizers
import torch
import torch.nn as nn
from tqdm import tqdm
import string
import csv

import transformers
from sklearn import model_selection
from transformers import AdamW
from transformers import get_linear_schedule_with_warmup


# 1. Config 

class config:
    MAX_LEN = 141
    TRAIN_BATCH_SIZE = 40 # 8 #??
    VALID_BATCH_SIZE = 16 #4 #??
    EPOCHS = 10
    BERT_PATH = "../input/bertbaseuncased/"
    MODEL_PATH = "model.bin"
    TRAINING_FILE = "../input/tweet-sentiment-extraction/train.csv"

    TOKENIZER = tokenizers.BertWordPieceTokenizer(
        os.path.join(BERT_PATH, 'vocab.txt'),
        lowercase=True
    )

    
    
# 2. Model

class BERTBaseUncased(nn.Module): #The parent class is nn.Module
    def __init__(self):
        super(BERTBaseUncased, self).__init__() #A call to super here ensures that both variables exist in the inherited class.
        self.bert = transformers.BertModel.from_pretrained(config.BERT_PATH)
        self.l0 = nn.Linear(768, 2) #output layer : fully_connected layer
        #(768, 2): by applying the bert model we will get 768 dimension embedding for each token, for the 
        # parameter '2', we set the value 2 because we need two logits in this case: start logits and end logits.
    
    def forward(self, ids, mask, token_type_ids):  #setiment can be the parameters here
        ''' See how to use sentiment in this part '''
        sequence_output, pooled_output = self.bert(
            ids, 
            attention_mask=mask,
            token_type_ids=token_type_ids
        )
        #sequence_output: (batch_size, num_tokens, 768) : every token has a vcetor of 768.
        # we want to go from the above output to: (batch_size, num_tokens, 2): only twi values associated with each token. 
        
        logits = self.l0(sequence_output)
        #shape of logits: (batch_size, num_tokens, 2)
        #I want to split the above output in two splitted parts: 
        # (batch_size, num_tokens, 1), (batch_size, num_tokens, 1).

        start_logits, end_logits = logits.split(1, dim=-1) # dim=-1 to seperate columns
        #shape: (batch_size, num_tokens, 1), (batch_size, num_tokens, 1).

        start_logits = start_logits.squeeze(-1) #it says where my selected_text is beginning
        end_logits = end_logits.squeeze(-1) #it says where my selected_text is ending, 1 : axis=1
        #shape: (batch_size, num_tokens), (batch_size, num_tokens).

        
        return start_logits, end_logits
    

    
# Device

device = torch.device("cuda") #??
model = BERTBaseUncased()
model.to(device)
model = nn.DataParallel(model)
model.load_state_dict(torch.load('../input/tweet-sentiment-extraction-training/model.bin'))

model.eval()

    
# Dataset
''
class TweetDataset:
    ''' initialization function '''
    def __init__(self, tweet, sentiment, selected_text):
        self.tweet = tweet
        self.sentiment = sentiment
        self.selected_text = selected_text
        self.max_len = config.MAX_LEN #max_len from config.py
        self.tokenizer = config.TOKENIZER #TOLENIZER from config.py

    ''' fct : length of the dataset '''
    def __len__(self): 
        return(len(self.tweet))

    def __getitem__(self, item): #item is the index value

        #remove all kind of spaces that we may have
        tweet = str(self.tweet[item]) # converting to string just to ckech that nothing breaks
        tweet = " ".join(tweet.split())

        #Doing the same for selected_text
        selected_text = str(self.selected_text[item]) # converting to string just to ckech that nothing breaks
        selected_text = " ".join(selected_text.split())

        ''' Prepare what we need to train our model '''
        #Length of selected text
        len_sel_text = len (selected_text)

        #
        idx0 = -1 
        idx1 = -1
        for ind in (i for i, e in enumerate(tweet) if e == selected_text[0]):
            if tweet[ind: ind+len_sel_text] == selected_text:
                idx0 = ind
                idx1 = ind + len_sel_text - 1 #I added '-1' because len takes index+1
                break
        
        #
        char_targets = [0] * len(tweet)
        #[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0] : characters
        if idx0 != -1 and idx1 != -1 :
            for j in range(idx0, idx1 + 1):
                if tweet[j] != ' ':  #if not space
                    char_targets[j] = 1
        #[0,0,0,0,1,1,1,0,1,1,1,0,0,1,1,1,1,0,1,1,0,0,0] : the 0s in the middle are spaces

        #
        tok_tweet = self.tokenizer.encode(tweet)
        tok_tweet_tokens = tok_tweet.tokens
        tok_tweet_ids = tok_tweet.ids
        tok_tweet_offsets = tok_tweet.offsets[1:-1] #because always the fisrt token is [CLS] and the last token is [SEP]

        #From characters to words: Define the targets variable: 
        targets = [0] * (len(tok_tweet_tokens)-2) # -2 to remove [CLS] and [SEP] tokens
        #[0,0,0,0,0,0,0]: words
        for j, (offset1, offset2) in enumerate(tok_tweet_offsets):
            if sum(char_targets[offset1:offset2]) > 0: #??
                #then the hole token has target of 1, to match partial words
                targets[j] = 1
        #[0,0,1,1,1,0,0]
        #=> we don't need char_targets anymore, we just need target.

        targets = [0] + targets + [0] #cls, sep
        targets_start = [0] * len(targets)
        targets_end = [0] *len(targets)

        #Create non zero variable:find the indices of the nonzero elements in targets
        non_zero = np.nonzero(targets)[0] 
        
        if len(non_zero) > 0: #if there are some values
            targets_start[non_zero[0]] = 1
            targets_end[non_zero[-1]] = 1 

        ''' Attention mask: a sequence of 1s and 0s, with 1s for all input tokens (actual words) and 0s
        for all padding tokens.'''
        mask = [1] * len(tok_tweet_ids)
        token_type_ids = [0] * len(tok_tweet_ids) #in case of two segments(texts), we need to differentiate them. We don't need it in our case.

        ''' 
        The convention in BERT is:
        (a) For sequence pairs:
        tokens:   [CLS] is this jack ##son ##ville ? [SEP] no it is not . [SEP]
        type_ids: 0     0  0    0    0     0       0 0     1  1  1  1   1 1 

        (b) For single sequences:
        tokens:   [CLS] the dog is hairy . [SEP]
        type_ids: 0     0   0   0  0     0 0

        '''
        
        padding_len = self.max_len - len(tok_tweet_ids)
        ''' pad everything you need: (all the outputs sould have the same length as max_len): '''
        ids = tok_tweet_ids + [0] * padding_len
        mask = mask + [0] * padding_len
        token_type_ids = token_type_ids + [0] * padding_len
        targets = targets + [0] * padding_len
        targets_start = targets_start + [0] * padding_len
        targets_end = targets_end + [0] * padding_len


        #Sentiment
        sentiment = [1, 0, 0] #neutral
        if self.sentiment[item] == 'positive':
            sentiment = [0, 0, 1] #one_hot encoding
        if self.sentiment[item] == 'negative':
            sentiment = [0, 1, 0]

        return{
            'ids': torch.tensor(ids, dtype=torch.long), #convert tp pytorch tensor
            'mask': torch.tensor(mask, dtype=torch.long),
            'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(targets, dtype=torch.long),
            'targets_start': torch.tensor(targets_start, dtype=torch.long),
            'targets_end': torch.tensor(targets_end, dtype=torch.long),
            'padding_len': torch.tensor(padding_len, dtype=torch.long),
            'tweet_tokens': ' '.join(tok_tweet_tokens),
            'orig_tweet': self.tweet[item],
            'sentiment': torch.tensor(sentiment, dtype=torch.long),
            'orig_sentiment': self.sentiment[item],
            'orig_selected_text': self.selected_text[item]
           
        }   


# Load the test file
df_test = pd.read_csv('../input/tweet-sentiment-extraction/test.csv')
df_test.loc[:, 'selected_text'] = df_test.text.values #just to make sur the dataset works without any change there
    
    
#Load the tweetDataset and create the loader
test_dataset = TweetDataset(
        tweet=df_test.text.values,
        sentiment=df_test.sentiment.values,
        selected_text=df_test.selected_text.values
)
    
valid_data_loader = torch.utils.data.DataLoader(
        test_dataset,
        shuffle = False,
        batch_size=config.VALID_BATCH_SIZE,
        num_workers=1
)    
    
#Evaluation
final_col = [] # to submit
fin_output_start = []
fin_output_end = []
fin_padding_lens = []
fin_tweet_tokens = []
fin_orig_sentiment = []
fin_orig_selected_text = []
fin_orig_tweet =[]


with torch.no_grad():
    for bi, d in enumerate(valid_data_loader): # bi : batch index, d : dataset
        ids = d["ids"]
        token_type_ids = d["token_type_ids"]
        mask = d["mask"]
        tweet_tokens =  d['tweet_tokens']
        padding_len = d['padding_len']
        orig_sentiment = d['orig_sentiment']
        orig_selected_text = d['orig_selected_text']
        orig_tweet = d['orig_tweet']
        targets_start = d["targets_start"]
        targets_end = d["targets_end"]


        ids = ids.to(device, dtype=torch.long)
        token_type_ids = token_type_ids.to(device, dtype=torch.long) #long is 64-bit integer
        mask = mask.to(device, dtype=torch.long)
        targets_start = targets_start.to(device, dtype=torch.float) #targets_start
        targets_end = targets_end.to(device, dtype=torch.float) #targets_end


        o1, o2 = model(
            ids=ids,
            mask=mask,
            token_type_ids=token_type_ids
        )

        # This converts the "logits" to "probability-like" scores
        fin_output_start.append(torch.sigmoid(o1).cpu().detach().numpy()) #??
        fin_output_end.append(torch.sigmoid(o2).cpu().detach().numpy())
        fin_padding_lens.extend(padding_len.cpu().detach().numpy().tolist()) #1 value so we can use extend

        fin_tweet_tokens.extend(tweet_tokens) #it is just a list of strings
        fin_orig_sentiment.extend(orig_sentiment)
        fin_orig_selected_text.extend(orig_selected_text)
        fin_orig_tweet.extend(orig_tweet)

    fin_output_start = np.vstack(fin_output_start)
    fin_output_end = np.vstack(fin_output_end)

    threshold = 0.2 #?? i its related to sigmoid??
    jaccards =[]
    for j in range(len(fin_tweet_tokens)):
        target_string = fin_orig_selected_text[j]
        tweet_tokens = fin_tweet_tokens[j]
        padding_len = fin_padding_lens[j]
        original_tweet = fin_orig_tweet[j]
        sentiment = fin_orig_sentiment[j]

        if padding_len > 0:
            mask_start = fin_output_start[j, :][:-padding_len] >= threshold #-padding_len??
            mask_end = fin_output_end[j, :][:-padding_len] >= threshold
        else:
            mask_start = fin_output_start[j, :] >= threshold
            mask_end = fin_output_end[j, :] >= threshold

        mask = [0] * len(mask_start)
        idx_start = np.nonzero(mask_start)[0]
        idx_end = np.nonzero(mask_end)[0]

        if len(idx_start) > 0:
            idx_start = idx_start[0]
            if len(idx_end) > 0:
                idx_end = idx_end[0]
            else:
                idx_end = idx_start
        else:
            idx_start = 0
            idx_end = 0

        for mj in range(idx_start, idx_end + 1):
            mask[mj] = 1

        output_tokens = [x for p, x in enumerate(tweet_tokens.split()) if mask[p] == 1]
        output_tokens = [x for x in output_tokens if x not in ('[CLS]', '[SEP]')]

        final_output = ''

        for ot in output_tokens:
            if ot.startswith('##'):
                final_output = final_output + ot[2:]
            elif len(ot) == 1 and ot in string.punctuation:
                final_output = final_output + ot
            else:
                final_output = final_output + ' ' + ot
        final_output = final_output.strip()
        if sentiment == 'neutral' or len(original_tweet.split()) < 4: #??
            final_output = original_tweet
        #final_col.append(final_output) # to submit  

        final_col.append(final_output) # to submit 

        
#Prepare csv file
submission = pd.read_csv('../input/tweet-sentiment-extraction/sample_submission.csv')
submission.selected_text = final_col
#submission.loc[:, 'selected_text'] = final_col
submission.to_csv('submission.csv',index=False)

print(submission.head())



    


ImportError: cannot import name 'AdamW' from 'transformers' (/usr/local/lib/python3.11/dist-packages/transformers/__init__.py)